In [1]:
# import libraries
import os
import json
import time
import boto3
import sagemaker

from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.parameters import ParameterString, ParameterFloat, ParameterInteger
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.execution_variables import ExecutionVariables

from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve
from sagemaker.model_metrics import ModelMetrics, MetricsSource
from sagemaker.model import ModelPackage

# AWS setup
region = boto3.Session().region_name
sess = sagemaker.Session()
pipeline_sess = PipelineSession()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
sm = boto3.client("sagemaker", region_name=region)

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)

# set endpoint
PROD_ENDPOINT_NAME = "crema-d-emotion-endpoint-2026-02-15-16-32"

# model group
MODEL_PACKAGE_GROUP = "crema-d-emotion-recognition-models"

# Local artifact input file
LOCAL_PROCESSED_CSV = "artifacts/emovo_features.csv"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Bucket: sagemaker-us-east-1-472875368112
Role: arn:aws:iam::472875368112:role/LabRole


In [2]:
%%writefile preprocess_emovo_unit_tests.py
import os
import json
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

FEATURES = [
    "rms_mean",
    "pitch_mean",
    "pitch_std",
    "spectral_centroid_mean",
    "mfcc_1_mean",
    "mfcc_2_mean",
    "mfcc_3_mean",
    "tempo",
]

def fail(msg: str):
    raise RuntimeError(msg)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-dir", type=str, default="/opt/ml/processing/input")
    parser.add_argument("--output-dir", type=str, default="/opt/ml/processing/output")
    parser.add_argument("--missing-threshold", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    in_csv = os.path.join(args.input_dir, "emovo_features.csv")
    if not os.path.exists(in_csv):
        cands = [f for f in os.listdir(args.input_dir) if f.endswith(".csv")]
        if not cands:
            fail(f"No CSV found in {args.input_dir}")
        in_csv = os.path.join(args.input_dir, cands[0])

    df = pd.read_csv(in_csv)

    # schema unit test
    if "emotion" not in df.columns:
        fail("Unit test failed: missing required column 'emotion'.")

    missing_feats = [c for c in FEATURES if c not in df.columns]
    if missing_feats:
        fail(f"Unit test failed: missing required feature columns: {missing_feats}")

    # tempo
    df["tempo"] = pd.to_numeric(df["tempo"], errors="coerce")

    # types unit test
    for c in FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # missingness checks
    miss = df[FEATURES].isna().mean().to_dict()
    offenders = {k: v for k, v in miss.items() if v > args.missing_threshold}
    if offenders:
        fail(f"Unit test failed: missingness above threshold {args.missing_threshold:.2%}: {offenders}")

    # remove NaNs
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["emotion"]).copy()

    if len(df) < 200:
        fail(f"Unit test failed: too few rows after cleaning ({len(df)}).")

    # label encoding
    le = LabelEncoder()
    df["emotion_encoded"] = le.fit_transform(df["emotion"].astype(str))

    # classes unit test
    n_classes = df["emotion_encoded"].nunique()
    if n_classes < 2:
        fail(f"Unit test failed: need >=2 classes, found {n_classes}.")

    os.makedirs(args.output_dir, exist_ok=True)

    # save mapping
    mapping = {cls: int(idx) for cls, idx in zip(le.classes_, le.transform(le.classes_))}
    with open(os.path.join(args.output_dir, "label_mapping.json"), "w") as f:
        json.dump(mapping, f, indent=2)

    # Use existing split if present, otherwise create split
    use_existing_split = "split" in df.columns and df["split"].isin(["train", "val", "test"]).all()

    if use_existing_split:
        train_df = df[df["split"] == "train"].copy()
        val_df   = df[df["split"] == "val"].copy()
        test_df  = df[df["split"] == "test"].copy()
        if min(len(train_df), len(val_df), len(test_df)) == 0:
            use_existing_split = False
    # use 80/10/10
    if not use_existing_split:
        tmp_train, test_df = train_test_split(
            df, test_size=0.10, random_state=args.seed, stratify=df["emotion_encoded"]
        )
        train_df, val_df = train_test_split(
            tmp_train, test_size=0.1111, random_state=args.seed, stratify=tmp_train["emotion_encoded"]
        )

    # XGBoost format
    def to_xgb(dfin: pd.DataFrame) -> pd.DataFrame:
        out = dfin[["emotion_encoded"] + FEATURES].copy()
        out = out.replace([np.inf, -np.inf], np.nan).dropna()
        return out

    train_xgb = to_xgb(train_df)
    val_xgb   = to_xgb(val_df)
    test_xgb  = to_xgb(test_df)

    # write outputs
    train_dir = os.path.join(args.output_dir, "train")
    val_dir   = os.path.join(args.output_dir, "validation")
    test_dir  = os.path.join(args.output_dir, "test")
    for d in [train_dir, val_dir, test_dir]:
        os.makedirs(d, exist_ok=True)

    train_xgb.to_csv(os.path.join(train_dir, "train.csv"), index=False, header=False)
    val_xgb.to_csv(os.path.join(val_dir, "validation.csv"), index=False, header=False)
    test_xgb.to_csv(os.path.join(test_dir, "test.csv"), index=False, header=False)

    # small report for debugging
    report = {
        "rows_in": int(len(df)),
        "rows_train": int(len(train_xgb)),
        "rows_val": int(len(val_xgb)),
        "rows_test": int(len(test_xgb)),
        "num_classes": int(n_classes),
        "missingness": miss,
        "labels": mapping,
    }
    with open(os.path.join(args.output_dir, "preprocess_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    print("Preprocess + unit tests passed.")
    print(json.dumps(report, indent=2))

if __name__ == "__main__":
    main()

Writing preprocess_emovo_unit_tests.py


In [21]:
%%writefile evaluate_xgb_emovo.py
import os
import json
import tarfile
import argparse
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score

def find_model_file(extract_dir: str) -> str:
    preferred = {"xgboost-model", "model.bin", "model"}
    found = []
    for root, _, files in os.walk(extract_dir):
        for f in files:
            full = os.path.join(root, f)
            found.append(full)
            if f in preferred:
                return full

    if len(found) == 1:
        return found[0]

    for full in found:
        if os.path.getsize(full) > 0:
            return full

    raise FileNotFoundError(f"Could not locate model file. Extracted files: {found[:25]}")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-artifact", type=str, required=True)
    parser.add_argument("--test-path", type=str, required=True)
    parser.add_argument("--output-dir", type=str, default="/opt/ml/processing/evaluation")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    # Extract model
    extract_dir = "/tmp/model_artifact"
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(args.model_artifact, "r:gz") as tar:
        tar.extractall(path=extract_dir)

    model_file = find_model_file(extract_dir)
    booster = xgb.Booster()
    booster.load_model(model_file)

    # test csv: label first
    test = pd.read_csv(args.test_path, header=None)
    y_true = test.iloc[:, 0].astype(int).values
    X = test.iloc[:, 1:].values

    dtest = xgb.DMatrix(X)
    y_pred = booster.predict(dtest)
    y_pred = np.array(y_pred, dtype=int)

    acc = float(accuracy_score(y_true, y_pred))

    evaluation = {"classification_metrics": {"accuracy": {"value": acc}}}

    out_path = os.path.join(args.output_dir, "evaluation.json")
    with open(out_path, "w") as f:
        json.dump(evaluation, f, indent=2)

    print("Saved:", out_path)
    print(json.dumps(evaluation, indent=2))

if __name__ == "__main__":
    main()


Overwriting evaluate_xgb_emovo.py


In [22]:
PREFIX_DEFAULT = "ser-cicd-emovo"
raw_key = f"{PREFIX_DEFAULT}/raw/emovo_features.csv"

boto3.client("s3", region_name=region).upload_file(LOCAL_PROCESSED_CSV, bucket, raw_key)
RAW_EMOVO_S3 = f"s3://{bucket}/{raw_key}"

print("Uploaded EMOVO features CSV to:", RAW_EMOVO_S3)

Uploaded EMOVO features CSV to: s3://sagemaker-us-east-1-472875368112/ser-cicd-emovo/raw/emovo_features.csv


In [17]:
# Pipeline parameters
p_prefix = ParameterString("Prefix", default_value=PREFIX_DEFAULT)
p_processing_instance = ParameterString("ProcessingInstanceType", default_value="ml.t3.xlarge")
p_training_instance   = ParameterString("TrainingInstanceType", default_value="ml.m5.xlarge")
p_accuracy_threshold  = ParameterFloat("AccuracyThreshold", default_value=0.70)

p_num_round = ParameterInteger("NumRound", default_value=100)
p_max_depth = ParameterInteger("MaxDepth", default_value=6)
p_eta       = ParameterFloat("Eta", default_value=0.3)

p_model_package_group = ParameterString("ModelPackageGroupName", default_value=MODEL_PACKAGE_GROUP)

# images/processors
xgb_image = retrieve(framework="xgboost", region=region, version="1.7-1")

sk_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=p_processing_instance,
    instance_count=1,
    sagemaker_session=pipeline_sess,
)

# preprocess and unit test
processing_base = Join(on="", values=[
    "s3://", bucket, "/", p_prefix, "/processing/", ExecutionVariables.PIPELINE_EXECUTION_ID
])

preprocess_step = ProcessingStep(
    name="PreprocessEmovoAndUnitTests",
    processor=sk_processor,
    code="preprocess_emovo_unit_tests.py",
    inputs=[
        ProcessingInput(
            source=Join(on="", values=["s3://", bucket, "/", p_prefix, "/raw/emovo_features.csv"]),
            destination="/opt/ml/processing/input",
            input_name="raw"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination=Join(on="", values=[processing_base, "/train"]),
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation",
            destination=Join(on="", values=[processing_base, "/validation"]),
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
            destination=Join(on="", values=[processing_base, "/test"]),
        ),
        ProcessingOutput(
            output_name="meta",
            source="/opt/ml/processing/output",
            destination=Join(on="", values=[processing_base, "/meta"]),
        ),
    ],
    job_arguments=[
        "--input-dir", "/opt/ml/processing/input",
        "--output-dir", "/opt/ml/processing/output",
        "--missing-threshold", "0.05",
        "--seed", "42",
    ],
)

train_s3 = preprocess_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
val_s3   = preprocess_step.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri
test_s3  = preprocess_step.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri

# train model
xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type=p_training_instance,
    volume_size=50,
    max_run=3600,
    output_path=Join(on="", values=["s3://", bucket, "/", p_prefix, "/training-output"]),
    sagemaker_session=pipeline_sess,
    base_job_name="ser-xgb-train-emovo",
)

xgb_estimator.set_hyperparameters(
    objective="multi:softmax",
    num_class=6,
    max_depth=p_max_depth,
    eta=p_eta,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    num_round=p_num_round,
)

train_step = TrainingStep(
    name="TrainXGBoostOnEmovo",
    estimator=xgb_estimator,
    inputs={
        "train": TrainingInput(s3_data=train_s3, content_type="text/csv"),
        "validation": TrainingInput(s3_data=val_s3, content_type="text/csv"),
    }
)

# evaluate model
eval_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=p_processing_instance,
    instance_count=1,
    sagemaker_session=pipeline_sess,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

evaluation_base = Join(on="", values=[
    "s3://", bucket, "/", p_prefix, "/evaluation/", ExecutionVariables.PIPELINE_EXECUTION_ID
])

eval_step = ProcessingStep(
    name="EvaluateCandidateOnEmovo",
    processor=eval_processor,
    code="evaluate_xgb_emovo.py",
    inputs=[
        ProcessingInput(
            source=train_step.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model/model.tar.gz",
            input_name="model"
        ),
        ProcessingInput(
            source=test_s3,
            destination="/opt/ml/processing/test",
            input_name="test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=evaluation_base
        )
    ],
    job_arguments=[
        "--model-artifact", "/opt/ml/processing/model/model.tar.gz",
        "--test-path", "/opt/ml/processing/test/test.csv",
        "--output-dir", "/opt/ml/processing/evaluation"
    ],
    property_files=[evaluation_report],
)

accuracy_val = JsonGet(
    step_name=eval_step.name,
    property_file=evaluation_report,
    json_path="classification_metrics.accuracy.value"
)

# check gate
fail_step = FailStep(
    name="FailIfAccuracyLow",
    error_message="Candidate model failed accuracy gate on EMOVO."
)

# register new model
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(on="", values=[evaluation_base, "/evaluation.json"]),
        content_type="application/json"
    )
)

register_step = RegisterModel(
    name="RegisterCandidateModel",
    estimator=xgb_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv", "application/json"],
    response_types=["text/csv", "application/json"],
    inference_instances=["ml.t2.medium", "ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.large", "ml.m5.xlarge"],
    model_package_group_name=p_model_package_group,
    approval_status="Approved",
    model_metrics=model_metrics,
)

cond_step = ConditionStep(
    name="AccuracyGate",
    conditions=[ConditionGreaterThanOrEqualTo(left=accuracy_val, right=p_accuracy_threshold)],
    if_steps=[register_step],
    else_steps=[fail_step],
)

pipeline_name = "ser-emovo-cicd-train-eval-register"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        p_prefix,
        p_processing_instance,
        p_training_instance,
        p_accuracy_threshold,
        p_num_round,
        p_max_depth,
        p_eta,
        p_model_package_group,
    ],
    steps=[preprocess_step, train_step, eval_step, cond_step],
    sagemaker_session=pipeline_sess,
)

print("Built pipeline:", pipeline_name)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Built pipeline: ser-emovo-cicd-train-eval-register


In [18]:
# Upsert and run pipeline
pipeline.upsert(role_arn=role)
print("Upserted pipeline:", pipeline_name)

execution = pipeline.start(parameters={
    "Prefix": PREFIX_DEFAULT,
    "ProcessingInstanceType": "ml.t3.xlarge",
    "TrainingInstanceType": "ml.m5.xlarge",
    "AccuracyThreshold": 0.40,
    "NumRound": 100,
    "MaxDepth": 6,
    "Eta": 0.3,
    "ModelPackageGroupName": MODEL_PACKAGE_GROUP
})

print("Execution ARN:", execution.arn)
execution.wait()
print("Execution status:", execution.describe()["PipelineExecutionStatus"])


Upserted pipeline: ser-emovo-cicd-train-eval-register
Execution ARN: arn:aws:sagemaker:us-east-1:472875368112:pipeline/ser-emovo-cicd-train-eval-register/execution/xiv5llwo36rg


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:17                                                                                   │
│                                                                                                  │
│   14 })                                                                                          │
│   15                                                                                             │
│   16 print("Execution ARN:", execution.arn)                                                      │
│ ❱ 17 execution.wait()                                                                            │
│   18 print("Execution status:", execution.describe()["PipelineExecutionStatus"])                 │
│   19                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:938 in wait               │
│                                                                                                  │
│    935 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    936 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    937 │   │   )                                                                                 │
│ ❱  938 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    939 │                                                                                         │
│    940 │   def result(self, step_name: str):                                                     │
│    941 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:55 in wait                            │
│                                                                                                  │
│    52 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    53 │   # method.                                                                              │
│    54 │   def wait(self, **kwargs):                                                              │
│ ❱  55 │   │   Waiter.wait(self, **kwargs)                                                        │
│    56 │                                                                                          │
│    57 │   wait.__doc__ = WaiterDocstring(                                                        │
│    58 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:374 in wait                           │
│                                                                                                  │
│   371 │   │   │   │   return                                                                     │
│   372 │   │   │   if current_state == 'failure':                                                 │
│   373 │   │   │   │   reason = f'Waiter encountered a terminal failure state: {acceptor.explan   │
│ ❱ 374 │   │   │   │   raise WaiterError(                                                         │
│   375 │   │   │   │   │   name=self.name,                                                        │
│   376 │   │   │   │   │   reason=reason,                                                         │
│   377 │   │   │   │   │   last_response=response,                                                │
╰────────────────────────────────────────────────────────────

In [19]:
# check error details
desc = execution.describe()
print("Status:", desc["PipelineExecutionStatus"])
print("FailureReason:", desc.get("FailureReason", "None"))
print("ExecutionArn:", desc["PipelineExecutionArn"])

steps = execution.list_steps()
for s in steps:
    print(f"{s['StepName']:<32} {s['StepStatus']}")

failed_steps = [s for s in steps if s["StepStatus"] in ("Failed", "Stopped")]
print("\nFailed/Stopped:", [s["StepName"] for s in failed_steps])

for s in failed_steps:
    print("\n---")
    print("StepName:", s["StepName"])
    print("StepStatus:", s["StepStatus"])
    print("FailureReason:", s.get("FailureReason", "None"))
    print("Metadata keys:", list((s.get("Metadata") or {}).keys()))


Status: Failed
FailureReason: Step failure: One or multiple steps failed.
ExecutionArn: arn:aws:sagemaker:us-east-1:472875368112:pipeline/ser-emovo-cicd-train-eval-register/execution/xiv5llwo36rg
EvaluateCandidateOnEmovo         Failed
TrainXGBoostOnEmovo              Succeeded
PreprocessEmovoAndUnitTests      Succeeded

Failed/Stopped: ['EvaluateCandidateOnEmovo']

---
StepName: EvaluateCandidateOnEmovo
StepStatus: Failed
FailureReason: ClientError: AlgorithmError: , exit code: 1
Metadata keys: ['ProcessingJob']


In [12]:
# evaluation json
exec_id = execution.arn.split("/")[-1]
eval_s3 = f"s3://{bucket}/{PREFIX_DEFAULT}/evaluation/{exec_id}/evaluation.json"
print("Evaluation JSON:", eval_s3)

Evaluation JSON: s3://sagemaker-us-east-1-472875368112/ser-cicd-emovo/evaluation/2o9pim0e3kon/evaluation.json


In [ ]:
# update model endpoint if model passes
status = execution.describe()["PipelineExecutionStatus"]
if status != "Succeeded":
    raise RuntimeError(f"Pipeline did not succeed (status={status}). Not promoting.")

# get current endpoint
ep_desc = sm.describe_endpoint(EndpointName=PROD_ENDPOINT_NAME)
old_config = ep_desc["EndpointConfigName"]
print("Current endpoint config:", old_config)

# get latest approved model
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=10
)["ModelPackageSummaryList"]

approved = [p for p in pkgs if p.get("ModelApprovalStatus") == "Approved"]
if not approved:
    raise RuntimeError("No Approved model packages found to promote.")

candidate_arn = approved[0]["ModelPackageArn"]
print("Promoting ModelPackageArn:", candidate_arn)

# create model package
mp = ModelPackage(
    role=role,
    model_package_arn=candidate_arn,
    sagemaker_session=sess
)

new_model_name = f"ser-emovo-candidate-{int(time.time())}"
create_model_resp = sm.create_model(
    ModelName=new_model_name,
    PrimaryContainer=mp.prepare_container_def(),
    ExecutionRoleArn=role
)
print("Created model:", new_model_name)

# create model endpoint
new_config_name = f"ser-emovo-config-{int(time.time())}"

old_cfg = sm.describe_endpoint_config(EndpointConfigName=old_config)
old_prod_variant = old_cfg["ProductionVariants"][0]
instance_type = old_prod_variant["InstanceType"]
initial_count = old_prod_variant["InitialInstanceCount"]

sm.create_endpoint_config(
    EndpointConfigName=new_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": new_model_name,
        "InitialInstanceCount": initial_count,
        "InstanceType": instance_type
    }]
)
print("Created endpoint config:", new_config_name)

# update existing endpoint
sm.update_endpoint(
    EndpointName=PROD_ENDPOINT_NAME,
    EndpointConfigName=new_config_name
)
print("UpdateEndpoint started. Waiting for InService...")

waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=PROD_ENDPOINT_NAME)

print("Endpoint is InService with new config:", new_config_name)
print("Rollback config (if needed):", old_config)
